# Data Loading Fundamentals

This notebook introduces the core concepts of loading text data into Neo4j for GraphRAG applications. You'll learn how to structure documents in a graph database to enable powerful retrieval patterns.

**Learning Objectives:**
- Understand the Document → Chunk graph structure
- Connect to Neo4j from a Jupyter notebook
- Create Document and Chunk nodes using Cypher
- Link chunks to documents and to each other with relationships

---

## Why Documents and Chunks?

When building GraphRAG applications, we split documents into smaller pieces called **chunks** because:

1. **Context windows are limited** - LLMs can only process a certain amount of text at once (e.g., 8K-200K tokens depending on the model)
2. **Retrieval precision** - Smaller chunks allow more precise matching to user queries. A 500-character chunk about "iPhone sales" is more relevant to a query about iPhones than an entire 50-page document.
3. **Cost efficiency** - Processing smaller chunks is faster and cheaper, both for embedding generation and LLM inference
4. **Embedding quality** - Embedding models produce better representations for focused, coherent text segments

The graph structure we'll build:
```
(:Document) <-[:FROM_DOCUMENT]- (:Chunk) -[:NEXT_CHUNK]-> (:Chunk)
```

This structure allows us to:
- **Track provenance**: Know which document each chunk came from
- **Preserve order**: Maintain the sequential reading order of chunks
- **Expand context**: Easily retrieve chunks before and after a matched chunk

---

## Text Splitting with neo4j-graphrag-python

We'll use `FixedSizeSplitter` from the [neo4j-graphrag-python](https://neo4j.com/docs/neo4j-graphrag-python/current/) library to split text into chunks:

- `chunk_size`: Maximum characters per chunk (e.g., 500 or 1000)
- `chunk_overlap`: Characters shared between consecutive chunks for context continuity. This ensures information at chunk boundaries isn't lost.
- `approximate=True` (default): Avoids splitting words mid-token by adjusting boundaries slightly

## Install Dependencies

First, install the required packages. This only needs to be run once per session.

In [ ]:
# Install neo4j-graphrag with Bedrock support
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" python-dotenv pydantic-settings nest-asyncio -q

## Setup

Import required modules and configure the environment.

In [ ]:
from data_utils import Neo4jConnection, DataLoader, split_text

## Sample Data

We'll load text from `company_data.txt` representing content from an SEC 10-K filing.

> **Note:** In production, you would use `pypdf` or similar libraries to extract text from PDF files. We use a pre-defined text file here for fast, reproducible results.

In [ ]:
# Load text from file using DataLoader
loader = DataLoader("company_data.txt")
SAMPLE_TEXT = loader.text

# Document metadata
DOCUMENT_PATH = "form10k-sample/apple-2023-10k.pdf"
DOCUMENT_PAGE = 1

metadata = loader.get_metadata()
print(f"Loaded from: {metadata['name']}")
print(f"Sample text length: {metadata['size']} characters")
print(f"\n{SAMPLE_TEXT}")

## Connect to Neo4j

Create a connection to your Neo4j database using the `Neo4jConnection` utility class.

In [ ]:
neo4j = Neo4jConnection().verify()
driver = neo4j.driver

## Clear Existing Data (Optional)

For a clean start, remove any existing Document and Chunk nodes from previous runs using the utility method.

In [ ]:
neo4j.clear_graph()

## Create Document Node

First, create a Document node to represent the source file. This node stores metadata about where the content came from, which is essential for:

- **Provenance tracking**: Knowing which file a chunk originated from
- **Filtering**: Searching only within specific documents
- **Citations**: Providing references in generated answers

We use `elementId(d)` to get the node's internal identifier, which we'll use to link chunks to this document.

In [ ]:
def create_document(driver, path: str, page: int) -> str:
    """Create a Document node and return its element ID."""
    with driver.session() as session:
        result = session.run("""
            CREATE (d:Document {path: $path, page: $page})
            RETURN elementId(d) as doc_id
        """, path=path, page=page)
        return result.single()["doc_id"]

doc_id = create_document(driver, DOCUMENT_PATH, DOCUMENT_PAGE)
print(f"Created Document node with ID: {doc_id}")

## Split Text into Chunks

Use `FixedSizeSplitter` from neo4j-graphrag-python to split the text into chunks with configurable size and overlap.

In [ ]:
# Split text using the utility function
chunks = split_text(SAMPLE_TEXT, chunk_size=500, chunk_overlap=50)

print(f"Split into {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {len(chunk)} chars")
    print(f"  {chunk[:100]}...\n")

## Create Chunk Nodes

Create Chunk nodes for each piece of text and link them to the Document with `FROM_DOCUMENT` relationships.

Each Chunk node stores:
- `text`: The actual chunk content
- `index`: The position in the original document (0, 1, 2, ...)

The `FROM_DOCUMENT` relationship points from the Chunk back to its source Document, enabling queries like "find all chunks from document X" or "which document does this chunk belong to?"

In [ ]:
def create_chunks(driver, doc_id: str, chunks: list[str]) -> list[str]:
    """Create Chunk nodes linked to a Document. Returns chunk element IDs."""
    chunk_ids = []
    with driver.session() as session:
        for index, text in enumerate(chunks):
            result = session.run("""
                MATCH (d:Document) WHERE elementId(d) = $doc_id
                CREATE (c:Chunk {text: $text, index: $index})
                CREATE (c)-[:FROM_DOCUMENT]->(d)
                RETURN elementId(c) as chunk_id
            """, doc_id=doc_id, text=text, index=index)
            chunk_id = result.single()["chunk_id"]
            chunk_ids.append(chunk_id)
            print(f"Created Chunk {index}")
    return chunk_ids

chunk_ids = create_chunks(driver, doc_id, chunks)
print(f"\nCreated {len(chunk_ids)} chunks")

## Link Chunks with NEXT_CHUNK

Create `NEXT_CHUNK` relationships between sequential chunks. This preserves the original document order and enables a powerful retrieval pattern: **context expansion**.

When you find a relevant chunk via vector search, the `NEXT_CHUNK` relationships let you easily retrieve:
- The **previous chunk** for background context
- The **next chunk** for continuation and conclusions

This is one of the key advantages of storing chunks in a graph rather than a simple vector store.

In [ ]:
def link_chunks(driver, chunk_ids: list[str]):
    """Create NEXT_CHUNK relationships between sequential chunks."""
    with driver.session() as session:
        for i in range(len(chunk_ids) - 1):
            session.run("""
                MATCH (c1:Chunk) WHERE elementId(c1) = $id1
                MATCH (c2:Chunk) WHERE elementId(c2) = $id2
                CREATE (c1)-[:NEXT_CHUNK]->(c2)
            """, id1=chunk_ids[i], id2=chunk_ids[i+1])
        print(f"Created {len(chunk_ids) - 1} NEXT_CHUNK relationships")

link_chunks(driver, chunk_ids)

## Verify the Graph Structure

Query the graph to see what we created.

In [ ]:
def show_graph_structure(driver):
    """Display the Document-Chunk graph structure."""
    with driver.session() as session:
        # Count nodes
        result = session.run("""
            MATCH (d:Document)
            OPTIONAL MATCH (d)<-[:FROM_DOCUMENT]-(c:Chunk)
            RETURN d.path as document, d.page as page, count(c) as chunks
        """)
        print("=== Graph Structure ===")
        for record in result:
            print(f"Document: {record['document']} (page {record['page']})")
            print(f"  Chunks: {record['chunks']}")
        
        # Show chunk chain
        result = session.run("""
            MATCH (c:Chunk)
            OPTIONAL MATCH (c)-[:NEXT_CHUNK]->(next:Chunk)
            RETURN c.index as idx, 
                   c.text as text,
                   next.index as next_idx
            ORDER BY c.index
        """)
        print("\n=== Chunk Chain ===")
        for record in result:
            next_str = f" -> Chunk {record['next_idx']}" if record['next_idx'] is not None else " (end)"
            print(f"Chunk {record['idx']}: \"{record['text']}\"{next_str}")

show_graph_structure(driver)

## Summary

In this notebook, you learned the foundational graph structure for GraphRAG applications:

1. **Document-Chunk structure** - Documents are split into smaller chunks for efficient retrieval. Each chunk is small enough to embed and retrieve precisely.

2. **FROM_DOCUMENT relationship** - Links chunks back to their source document, enabling provenance tracking and document-level filtering.

3. **NEXT_CHUNK relationship** - Preserves the sequential order of chunks, enabling context expansion during retrieval.

This basic structure is the foundation for all GraphRAG applications. The graph structure gives us capabilities that simple vector stores don't have - namely, the ability to traverse relationships to gather additional context.

In the next notebook, you'll learn to add **embeddings** to these chunks, enabling semantic similarity search.

---

**Next:** [Embeddings and Vector Search](02_embeddings.ipynb)

In [ ]:
# Cleanup
neo4j.close()